# MRO Parts Catalog — Exploratory Data Analysis

This notebook is for investigation only. Nothing here modifies the data.  
Decisions made here get implemented in `cleaning_pipeline.ipynb`.

**Loads:** `data/raw_data.csv` (raw) or `data/cleaned_data.csv` (post-pipeline)

---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)

print('Imports OK')

---
## 1. Load
Switch between raw and cleaned as needed.

In [ ]:
# Toggle between raw and cleaned
MODE = 'raw'   # 'raw' or 'clean'

if MODE == 'raw':
    df = pd.read_csv('data/raw_data.csv', dtype=str, low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
else:
    df = pd.read_csv('data/cleaned_data.csv', low_memory=False)

print(f'Mode: {MODE}')
print(f'Shape: {df.shape}')

---
## 2. Null Audit
Full breakdown of every column — fill rate, unique count, sample values.

In [ ]:
audit = pd.DataFrame({
    'pct_null':   (df.isna().mean() * 100).round(2),
    'null_count': df.isna().sum(),
    'n_unique':   df.nunique(),
    'dtype':      df.dtypes.astype(str),
}).sort_values('pct_null', ascending=False).reset_index().rename(columns={'index': 'column'})

audit

---
## 3. Inspect Any Column
Quick tool — change `COL` to inspect any column in detail.

In [ ]:
COL = 'attribute_table'   # <-- change this

filled   = df[COL].notna().sum()
n_unique = df[COL].nunique()
system_n = (df[COL] == 'system').sum() if df[COL].dtype == object else 0

print(f'{COL}')
print(f'  filled:        {filled:,} / {len(df):,} ({filled/len(df)*100:.1f}%)')
print(f'  unique:        {n_unique:,}')
print(f'  just system:   {system_n:,}')
print()

if n_unique <= 20:
    print('Value counts:')
    print(df[COL].value_counts(dropna=True).to_string())
else:
    print('Sample values:')
    for v in df[COL].dropna().head(5):
        print(f'  → {str(v)[:200]}')

---
## 4. Compare Two Columns Side by Side
Use to check overlap, identity, or meaningful difference between column pairs.

In [ ]:
COL_A = 'descriptions.mro.value'       # <-- change
COL_B = 'descriptions.livhaven.value'  # <-- change

has_both = df[COL_A].notna() & df[COL_B].notna()
both     = df[has_both][[COL_A, COL_B, 'manufacturer_part_number']].copy()
identical = (both[COL_A] == both[COL_B]).sum()

print(f'Rows with both filled: {len(both):,}')
print(f'  Identical: {identical:,} ({identical/len(both)*100:.1f}%)')
print(f'  Different: {len(both)-identical:,} ({(len(both)-identical)/len(both)*100:.1f}%)')
print(f'\nOnly {COL_A}: {(df[COL_A].notna() & df[COL_B].isna()).sum():,}')
print(f'Only {COL_B}: {(df[COL_A].isna() & df[COL_B].notna()).sum():,}')
print(f'Neither:     {(df[COL_A].isna() & df[COL_B].isna()).sum():,}')

print('\n5 rows where they differ:')
diff = both[both[COL_A] != both[COL_B]].head(5)
for idx, row in diff.iterrows():
    print(f"\n  Part: {row.get('manufacturer_part_number', idx)}")
    print(f"  A: {str(row[COL_A])[:150]}")
    print(f"  B: {str(row[COL_B])[:150]}")

---
## 5. Brand / Category Distribution

In [ ]:
brand_col    = 'brand.default.title' if MODE == 'raw' else 'brand_name'
category_col = 'category.default.title' if MODE == 'raw' else 'category_name'

print('Top 20 brands:')
print(df[brand_col].value_counts().head(20).to_string())

print('\nTop 20 categories:')
print(df[category_col].value_counts().head(20).to_string())

---
## 6. Numeric Distributions

In [ ]:
price_col  = 'last_sold_price'
weight_col = 'item_weight' if MODE == 'clean' else 'itemweight'

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for col, ax_lin, ax_log in [
    (price_col,  axes[0][0], axes[0][1]),
    (weight_col, axes[1][0], axes[1][1]),
]:
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    s = s[s > 0]
    ax_lin.hist(s, bins=50)
    ax_lin.set_title(f'{col} (linear)')
    ax_log.hist(np.log10(s), bins=50)
    ax_log.set_title(f'{col} (log10)')

plt.tight_layout()
plt.show()

---
## 7. Attribute Table — Key Coverage
Run on cleaned data to see which spec attributes are most populated.

In [ ]:
# Only useful post-cleaning when attr_ columns exist
attr_cols = [c for c in df.columns if c.startswith('attr_')]
if not attr_cols:
    print('No attr_ columns found — run on cleaned data.')
else:
    coverage = df[attr_cols].notna().sum().sort_values(ascending=False)
    print(f'Total attribute columns: {len(attr_cols)}')
    print('\nTop 30 by row coverage:')
    print(coverage.head(30).to_string())

---
## 8. Part Number Format Analysis

In [ ]:
pn_col = 'manufacturer_part_number'
pn = df[pn_col].dropna().astype(str)

print(f'Total non-null: {len(pn):,}')
print(f'Has spaces:     {pn.str.contains(" ").sum():,}')
print(f'Has lowercase:  {pn.str.contains("[a-z]", regex=True).sum():,}')
print(f'Length < 4:     {(pn.str.len() < 4).sum():,}')
print(f'Length > 30:    {(pn.str.len() > 30).sum():,}')
print(f'\nLength distribution:')
print(pn.str.len().describe())

---
## 9. Scratch Cell
Free-form investigation. Copy/paste from here when something needs a permanent home.

In [ ]:
# scratch
